In [1]:
#Import Libraries
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
#Define Paths
RAW_PATH = "../data/raw/"
PROCESSED_PATH = "../data/processed/"

os.makedirs(PROCESSED_PATH, exist_ok=True)

print("Raw path:", RAW_PATH)
print("Processed path:", PROCESSED_PATH)

Raw path: ../data/raw/
Processed path: ../data/processed/


Load All Datasets

In [3]:

customers = pd.read_csv(
    RAW_PATH + "olist_customers_dataset.csv"
)

geolocation = pd.read_csv(
    RAW_PATH + "olist_geolocation_dataset.csv"
)

order_items = pd.read_csv(
    RAW_PATH + "olist_order_items_dataset.csv"
)

payments = pd.read_csv(
    RAW_PATH + "olist_order_payments_dataset.csv"
)

reviews = pd.read_csv(
    RAW_PATH + "olist_order_reviews_dataset.csv"
)

orders = pd.read_csv(
    RAW_PATH + "olist_orders_dataset.csv"
)

products = pd.read_csv(
    RAW_PATH + "olist_products_dataset.csv"
)

sellers = pd.read_csv(
    RAW_PATH + "olist_sellers_dataset.csv"
)

translation = pd.read_csv(
    RAW_PATH + "product_category_name_translation.csv"
)

print("All datasets loaded successfully!")

All datasets loaded successfully!


Create Dataset Dictionary

In [4]:
datasets = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "translation": translation
}

print(f"Total datasets: {len(datasets)}")

Total datasets: 9


Check Duplicates


First, let's see what we're dealing with.

In [5]:
for name, df in datasets.items():
    duplicates = df.duplicated().sum()
    
    print(
        f"{name:<15} → "
        f"{duplicates:,} duplicate rows"
    )

customers       → 0 duplicate rows
geolocation     → 261,831 duplicate rows
order_items     → 0 duplicate rows
payments        → 0 duplicate rows
reviews         → 0 duplicate rows
orders          → 0 duplicate rows
products        → 0 duplicate rows
sellers         → 0 duplicate rows
translation     → 0 duplicate rows


Remove Exact Duplicate Rows

In [6]:
customers = customers.drop_duplicates()
orders = orders.drop_duplicates()
order_items = order_items.drop_duplicates()
payments = payments.drop_duplicates()
reviews = reviews.drop_duplicates()
products = products.drop_duplicates()
sellers = sellers.drop_duplicates()
translation = translation.drop_duplicates()

Convert Order Dates

In [7]:
order_date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in order_date_columns:
    orders[col] = pd.to_datetime(
        orders[col],
        errors="coerce"
    )

In [8]:
orders[order_date_columns].dtypes

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

Convert Shipping Date

In [9]:
order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"],
    errors="coerce"
)

order_items["shipping_limit_date"].dtype

dtype('<M8[ns]')

In [10]:
orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Validate Order Status vs Delivery Dates

This helps us understand why delivery dates are missing.

In [11]:
pd.crosstab(
    orders["order_status"],
    orders["order_delivered_customer_date"].isnull()
)

order_delivered_customer_date,False,True
order_status,,
approved,0,2
canceled,6,619
created,0,5
delivered,96470,8
invoiced,0,314
processing,0,301
shipped,0,1107
unavailable,0,609


Product Missing Values

You'll find missing values in some product dimension columns.

In [12]:
products.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

Clean Product Category

In [13]:
products["product_category_name"].isnull().sum()

np.int64(610)

In [14]:
products["product_category_name"] = (
    products["product_category_name"]
    .fillna("unknown")
)

Translate Product Categories

The dataset provides a Portuguese → English category mapping.

In [15]:
products = products.merge(
    translation,
    on="product_category_name",
    how="left"
)

In [16]:
products[
    [
        "product_category_name",
        "product_category_name_english"
    ]
].head(10)

,product_category_name,product_category_name_english
0,perfumaria,perfumery
1,artes,art
2,esporte_lazer,sports_leisure
3,bebes,baby
4,utilidades_domesticas,housewares
5,instrumentos_musicais,musical_instruments
6,cool_stuff,cool_stuff
7,moveis_decoracao,furniture_decor
8,eletrodomesticos,home_appliances
9,brinquedos,toys


Create Final Product Category

Use English where available.

In [17]:
products["product_category"] = (
    products["product_category_name_english"]
    .fillna(products["product_category_name"])
    .fillna("unknown")
)

Clean Customer IDs

In [18]:
customers[
    [
        "customer_id",
        "customer_unique_id"
    ]
].head()

,customer_id,customer_unique_id
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066


Check Primary Key Duplicates

In [19]:
orders["order_id"].duplicated().sum()

np.int64(0)

In [20]:
customers["customer_id"].duplicated().sum()

np.int64(0)

In [21]:
products["product_id"].duplicated().sum()

np.int64(0)

In [22]:
sellers["seller_id"].duplicated().sum()

np.int64(0)

Check Order Items

order_id itself can repeat in order_items because one order can contain multiple products.

In [23]:
order_items["order_id"].duplicated().sum()

np.int64(13984)

In [24]:
#This is a much better data-quality check.
order_items.duplicated(
    subset=["order_id", "order_item_id"]
).sum()

np.int64(0)

Check Foreign-Key Integrity

This is a very important step for our future MySQL database.

Order → Customer

In [25]:
invalid_customer_ids = (
    ~orders["customer_id"].isin(
        customers["customer_id"]
    )
)

print(
    "Invalid customer references:",
    invalid_customer_ids.sum()
)

Invalid customer references: 0


In [26]:
#Order Items → Orders

invalid_order_ids = (
    ~order_items["order_id"].isin(
        orders["order_id"]
    )
)

print(
    "Invalid order references:",
    invalid_order_ids.sum()
)

Invalid order references: 0


In [27]:
#Order Items → Products


invalid_product_ids = (
    ~order_items["product_id"].isin(
        products["product_id"]
    )
)

print(
    "Invalid product references:",
    invalid_product_ids.sum()
)

Invalid product references: 0


In [28]:
#Order Items → Sellers

invalid_seller_ids = (
    ~order_items["seller_id"].isin(
        sellers["seller_id"]
    )
)

print(
    "Invalid seller references:",
    invalid_seller_ids.sum()
)

Invalid seller references: 0


Create Delivery Time

Now we're moving from cleaning → feature engineering.

In [33]:
# Calculate actual delivery time in days

orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / (60 * 60 * 24)

print("delivery_days column created successfully.")

orders["delivery_days"].describe()

delivery_days column created successfully.


count    96476.000000
mean        12.558702
std          9.546530
min          0.533414
25%          6.766403
50%         10.217755
75%         15.720327
max        209.628611
Name: delivery_days, dtype: float64

In [34]:
# Calculate how many days early/late the order was delivered
# Positive value  = late
# Negative value  = early
# Zero            = exactly on estimated date

orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.total_seconds() / (60 * 60 * 24)

print("delivery_delay_days column created successfully.")

orders["delivery_delay_days"].describe()

delivery_delay_days column created successfully.


count    96476.000000
mean       -11.179120
std         10.186113
min       -146.016123
25%        -16.244384
50%        -11.948941
75%         -6.390000
max        188.975081
Name: delivery_delay_days, dtype: float64

Create Late Delivery Flag

In [35]:
# Create a True/False flag for late deliveries

orders["is_late"] = (
    orders["delivery_delay_days"] > 0
)

print("is_late column created successfully.")

print(orders["is_late"].value_counts(dropna=False))

is_late column created successfully.
is_late
False    91614
True      7827
Name: count, dtype: int64


In [36]:
orders[
    [
        "order_id",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_days",
        "delivery_delay_days",
        "is_late"
    ]
].head(10)

,order_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,8.436574,-7.107488,False
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,13.782037,-5.355729,False
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,9.394213,-17.245498,False
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,13.208750,-12.980069,False
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,2.873877,-9.238171,False
5,a4591c265e18cb1dcee52889e2d8acc3,2017-07-09 21:57:05,2017-07-26 10:57:55,2017-08-01,16.542245,-5.543113,False
6,136cce7faa42fdb2cefd53fdc79a6098,2017-04-11 12:22:08,NaT,2017-05-09,NaN,NaN,False
7,6514b8ad8028c9f2cc2374ded245783f,2017-05-16 13:10:30,2017-05-26 12:55:51,2017-06-07,9.989826,-11.461215,False
8,76c6e866289321a7c93b82b54852dc33,2017-01-23 18:29:09,2017-02-02 14:08:10,2017-03-06,9.818762,-31.410995,False
9,e69bfb5eb88e0ed6a785585b27e16dbf,2017-07-29 11:55:02,2017-08-16 17:14:30,2017-08-23,18.221852,-6.281597,False


Create Order Revenue

We'll eventually join order items with orders.

In [37]:
order_revenue = (
    order_items
    .groupby("order_id")
    .agg(
        product_revenue=("price", "sum"),
        freight_cost=("freight_value", "sum"),
        total_items=("order_item_id", "count")
    )
    .reset_index()
)

In [38]:
order_revenue.head()

,order_id,product_revenue,freight_cost,total_items
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,1
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,1
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,1
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,1


Merge Revenue With Orders

In [39]:
orders_analytics = orders.merge(
    order_revenue,
    on="order_id",
    how="left"
)

In [40]:
orders_analytics.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days,is_late,product_revenue,freight_cost,total_items
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.436574,-7.107488,False,29.99,8.72,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.782037,-5.355729,False,118.70,22.76,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.394213,-17.245498,False,159.90,19.22,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.208750,-12.980069,False,45.00,27.20,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.873877,-9.238171,False,19.90,8.72,1.0


Calculate Total Order Value

In [41]:

orders_analytics["total_order_value"] = (
    orders_analytics["product_revenue"]
    + orders_analytics["freight_cost"]
)

Create Date Features

In [42]:
orders_analytics["order_date"] = (
    orders_analytics["order_purchase_timestamp"].dt.date
)

orders_analytics["order_year"] = (
    orders_analytics["order_purchase_timestamp"].dt.year
)

orders_analytics["order_month"] = (
    orders_analytics["order_purchase_timestamp"].dt.month
)

orders_analytics["order_month_name"] = (
    orders_analytics["order_purchase_timestamp"].dt.month_name()
)

orders_analytics["order_quarter"] = (
    orders_analytics["order_purchase_timestamp"].dt.quarter
)

orders_analytics["order_day"] = (
    orders_analytics["order_purchase_timestamp"].dt.day
)

orders_analytics["order_day_of_week"] = (
    orders_analytics["order_purchase_timestamp"].dt.day_name()
)

Create Monthly Sales Dataset

In [43]:
monthly_sales = (
    orders_analytics[
        orders_analytics["order_status"] == "delivered"
    ]
    .groupby(
        ["order_year", "order_month"]
    )
    .agg(
        revenue=("product_revenue", "sum"),
        freight=("freight_cost", "sum"),
        orders=("order_id", "nunique"),
        items=("total_items", "sum")
    )
    .reset_index()
)

In [44]:
#Create a proper date:

monthly_sales["date"] = pd.to_datetime(
    monthly_sales["order_year"].astype(str)
    + "-"
    + monthly_sales["order_month"].astype(str)
    + "-01"
)

In [45]:
#Sort:

monthly_sales = monthly_sales.sort_values("date")

Inspect Monthly Sales


In [46]:
monthly_sales.head(10)

,order_year,order_month,revenue,freight,orders,items,date
0,2016,9,134.97,8.49,1,3.0,2016-09-01
1,2016,10,40325.11,6165.55,265,313.0,2016-10-01
2,2016,12,10.90,8.72,1,1.0,2016-12-01
3,2017,1,111798.36,15684.01,750,913.0,2017-01-01
4,2017,2,234223.40,37015.92,1653,1858.0,2017-02-01
5,2017,3,359198.85,55132.10,2546,2897.0,2017-03-01
6,2017,4,340669.68,50142.72,2303,2569.0,2017-04-01
7,2017,5,489338.25,77513.15,3546,4004.0,2017-05-01
8,2017,6,421923.37,68127.00,3135,3489.0,2017-06-01
9,2017,7,481604.52,84694.56,3872,4416.0,2017-07-01


In [47]:
monthly_sales.tail(10)

,order_year,order_month,revenue,freight,orders,items,date
13,2017,11,987765.37,165598.83,7289,8475.0,2017-11-01
14,2017,12,726033.19,117045.10,5513,6187.0,2017-12-01
15,2018,1,924645.00,153242.46,7069,8037.0,2018-01-01
16,2018,2,826437.13,139731.28,6555,7518.0,2018-02-01
17,2018,3,953356.25,167241.99,7003,8017.0,2018-03-01
18,2018,4,973534.09,159344.84,6798,7827.0,2018-04-01
19,2018,5,977544.69,151229.83,6749,7810.0,2018-05-01
20,2018,6,856077.86,155900.43,6099,7010.0,2018-06-01
21,2018,7,867953.46,159853.82,6159,6963.0,2018-07-01
22,2018,8,838576.64,146915.00,6351,7142.0,2018-08-01


Create Customer-Level Dataset

In [48]:
#First connect orders to customers.

customer_orders = orders_analytics.merge(
    customers[
        [
            "customer_id",
            "customer_unique_id",
            "customer_city",
            "customer_state"
        ]
    ],
    on="customer_id",
    how="left"
)

Customer RFM Dataset

In [49]:
delivered_orders = customer_orders[
    customer_orders["order_status"] == "delivered"
].copy()

In [50]:
reference_date = (
    delivered_orders["order_purchase_timestamp"].max()
    + pd.Timedelta(days=1)
)

rfm = (
    delivered_orders
    .groupby("customer_unique_id")
    .agg(
        last_purchase=(
            "order_purchase_timestamp",
            "max"
        ),
        frequency=(
            "order_id",
            "nunique"
        ),
        monetary=(
            "product_revenue",
            "sum"
        )
    )
    .reset_index()
)

In [51]:
#Calculate recency:

rfm["recency"] = (
    reference_date
    - rfm["last_purchase"]
).dt.days

RFM result

In [52]:
rfm = rfm[
    [
        "customer_unique_id",
        "recency",
        "frequency",
        "monetary"
    ]
]

rfm.head()

,customer_unique_id,recency,frequency,monetary
0,0000366f3b9a7992bf8c76cfdf3221e2,112,1,129.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,115,1,18.90
2,0000f46a3911fa3c0805444483337064,537,1,69.00
3,0000f6ccb0745a6a4b88665a16c9f078,321,1,25.99
4,0004aac84e0df4da2b147fca70cf8255,288,1,180.00


Save Cleaned Core Tables

In [53]:
customers.to_csv(
    PROCESSED_PATH + "customers_clean.csv",
    index=False
)

orders_analytics.to_csv(
    PROCESSED_PATH + "orders_analytics.csv",
    index=False
)

order_items.to_csv(
    PROCESSED_PATH + "order_items_clean.csv",
    index=False
)

payments.to_csv(
    PROCESSED_PATH + "payments_clean.csv",
    index=False
)

reviews.to_csv(
    PROCESSED_PATH + "reviews_clean.csv",
    index=False
)

products.to_csv(
    PROCESSED_PATH + "products_clean.csv",
    index=False
)

sellers.to_csv(
    PROCESSED_PATH + "sellers_clean.csv",
    index=False
)

In [54]:
#Save Analytical Datasets
monthly_sales.to_csv(
    PROCESSED_PATH + "monthly_sales.csv",
    index=False
)

rfm.to_csv(
    PROCESSED_PATH + "customer_rfm.csv",
    index=False
)

print("All processed datasets saved successfully!")

All processed datasets saved successfully!


In [55]:
print("Processed files:")

for file in os.listdir(PROCESSED_PATH):
    print("-", file)

Processed files:
- customers_clean.csv
- customer_rfm.csv
- data_quality_summary.csv
- kpi_summary.csv
- monthly_sales.csv
- orders_analytics.csv
- order_items_clean.csv
- payments_clean.csv
- products_clean.csv
- reviews_clean.csv
- sellers_clean.csv


                 RAW DATA
                    │
                    ▼
             Data Cleaning
                    │
        ┌───────────┼───────────┐
        ↓           ↓           ↓
    Validation   Cleaning   Feature Engineering
        │           │           │
        └───────────┼───────────┘
                    ↓
              CLEAN DATA
                    │
        ┌───────────┼────────────┐
        ↓           ↓            ↓
      MySQL      Power BI       ML
        │                        │
        └──────────┬─────────────┘
                   ↓
               AI AGENTS